# 3.3 SES Prototype — Jamming / Interference (Reproducible GitHub Version)
**Purpose:** Build a lightweight anomaly detector for communication anomalies (proxy for jamming/interference), generate the dashboard artefacts (SHAP CSV + heatmaps), and persist alert windows for the Streamlit app.

**Inputs (repo):**
- `data/interim/ses_comm_features.csv` (preferred if present)
- or `data/processed/ses_comm_features.csv` (if you have moved it)

**Outputs (repo):**
- `reports/figures/jamming_event_shap_values.csv`
- `reports/figures/jamming_event_heatmap.png`
- `reports/figures/jamming_continuous_heatmap.png`
- `data/processed/jam_test_eventized_scores.csv`

Notes:
- This notebook is designed to be run from `notebooks/` in the GitHub repo.
- SHAP explanations are produced via a **surrogate model** (Random Forest) trained to approximate the anomaly score.


In [ ]:
# ============================================================
# 0) Imports
# ============================================================
from __future__ import annotations

from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest, RandomForestRegressor

import matplotlib.pyplot as plt

import shap


In [ ]:
# ============================================================
# 1) Resolve repo paths (DO NOT change repo tree)
# ============================================================
HERE = Path.cwd().resolve()
REPO = HERE
while not (REPO / "app.py").exists() and REPO != REPO.parent:
    REPO = REPO.parent

assert (REPO / "app.py").exists(), f"Repo root not found. Current working dir: {HERE}"

DATA_PROCESSED = REPO / "data" / "processed"
DATA_INTERIM   = REPO / "data" / "interim"
FIG_DIR        = REPO / "reports" / "figures"

DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Prefer interim for features (based on your repo tree); fall back to processed if you moved it
IN_FEATURES = (DATA_INTERIM / "ses_comm_features.csv")
if not IN_FEATURES.exists():
    IN_FEATURES = (DATA_PROCESSED / "ses_comm_features.csv")

print("REPO          :", REPO)
print("DATA_PROCESSED:", DATA_PROCESSED)
print("DATA_INTERIM  :", DATA_INTERIM)
print("FIG_DIR       :", FIG_DIR)
print("IN_FEATURES   :", IN_FEATURES)


In [ ]:
# ============================================================
# 2) Load communication features
# ============================================================
if not IN_FEATURES.exists():
    raise FileNotFoundError(
        f"Missing input file: {IN_FEATURES}.\n"
        "Expected one of:\n"
        "- data/interim/ses_comm_features.csv\n"
        "- data/processed/ses_comm_features.csv"
    )

df = pd.read_csv(IN_FEATURES)

# Ensure time column exists and is parsed
time_candidates = [c for c in df.columns if c.lower() in ("time", "timestamp", "datetime", "date")]
if not time_candidates:
    raise ValueError("Input features CSV must contain a time-like column (e.g., 'time').")

tcol = time_candidates[0]
df[tcol] = pd.to_datetime(df[tcol], errors="coerce", utc=True)

# Optional ID columns if present (beam, modem, etc.)
id_col = None
for cand in ("beam", "beam_id", "satellite", "modem", "carrier", "link_id"):
    if cand in df.columns:
        id_col = cand
        break

# Feature columns: drop time and any obvious identifiers
drop = {tcol}
if id_col:
    drop.add(id_col)

feature_cols = [c for c in df.columns if c not in drop]

X = df[feature_cols].copy()
X = X.replace([np.inf, -np.inf], np.nan).fillna(method="ffill").fillna(0)

print("Rows:", len(df))
print("Features:", len(feature_cols))
print("Time column:", tcol)
print("ID column:", id_col)


In [ ]:
# ============================================================
# 3) Unsupervised anomaly scoring (proxy for interference)
# ============================================================
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X.values)

iso = IsolationForest(
    n_estimators=400,
    contamination="auto",
    random_state=42,
    n_jobs=-1
)
iso.fit(X_scaled)

raw = -iso.decision_function(X_scaled)
raw = (raw - raw.min()) / (raw.max() - raw.min() + 1e-12)

df_scores = pd.DataFrame({
    "time": df[tcol],
    "anomaly_score": raw,
})
if id_col:
    df_scores["beam"] = df[id_col].astype(str)
else:
    df_scores["beam"] = "COMM"

# Sort by time for eventization
df_scores = df_scores.sort_values("time").reset_index(drop=True)

print("Anomaly score summary:")
print(df_scores["anomaly_score"].describe())


In [ ]:
# ============================================================
# 4) Eventization -> data/processed/jam_test_eventized_scores.csv
# ============================================================
# Conservative: flag top 1% as anomalous windows
thr = float(np.quantile(df_scores["anomaly_score"].values, 0.99))
df_scores["is_anomaly"] = df_scores["anomaly_score"] >= thr

# Group consecutive anomaly windows into events per beam
events = []
for beam, grp_df in df_scores.groupby("beam"):
    g = grp_df.copy()
    g["grp"] = (g["is_anomaly"] != g["is_anomaly"].shift(1)).cumsum()
    for _, chunk in g.groupby("grp"):
        if not bool(chunk["is_anomaly"].iloc[0]):
            continue
        t_start = chunk["time"].iloc[0]
        t_end = chunk["time"].iloc[-1]
        events.append({
            "beam": beam,
            "t_start": t_start,
            "t_end": t_end,
            "label": 1,
            "severity": "high",
            "anomaly_score_max": float(chunk["anomaly_score"].max()),
        })

events_df = pd.DataFrame(events).sort_values(["t_start", "beam"]).reset_index(drop=True)
OUT_EVENTS = DATA_PROCESSED / "jam_test_eventized_scores.csv"
events_df.to_csv(OUT_EVENTS, index=False)

print("Threshold (99th percentile):", thr)
print("Events:", len(events_df))
print("Saved:", OUT_EVENTS)


## SHAP artefacts for the dashboard
The Streamlit dashboard loads the SHAP matrix CSV from:
`reports/figures/jamming_event_shap_values.csv`.

It will also display heatmap PNGs if present:
- `reports/figures/jamming_event_heatmap.png`
- `reports/figures/jamming_continuous_heatmap.png`

**Caveat:** SHAP explanations here are produced using a surrogate Random Forest model trained to approximate the anomaly score.

In [ ]:
# ============================================================
# 5) SHAP helper: save matrix CSV in the format expected by app.py
#    index = features, columns = time steps
# ============================================================
def save_shap_matrix_csv(out_csv: Path, shap_matrix_2d: np.ndarray, feature_names: list[str], time_labels: list[str] | None = None) -> None:
    out_csv = Path(out_csv)
    out_csv.parent.mkdir(parents=True, exist_ok=True)

    arr = np.asarray(shap_matrix_2d, dtype=float)

    if arr.ndim != 2:
        raise ValueError(f"Expected 2D matrix. Got shape={arr.shape}")

    if arr.shape[0] != len(feature_names):
        raise ValueError(
            f"Shape mismatch: matrix has {arr.shape[0]} rows but {len(feature_names)} features."
        )

    if time_labels is None:
        time_labels = [f"t{i}" for i in range(arr.shape[1])]

    df_out = pd.DataFrame(arr, index=feature_names, columns=time_labels)
    df_out.to_csv(out_csv)
    print(f"Saved SHAP matrix CSV -> {out_csv}")


In [ ]:
# ============================================================
# 6) Train surrogate + compute event SHAP matrix (features x time)
# ============================================================
# Surrogate regressor: explains anomaly_score in terms of features
rf = RandomForestRegressor(
    n_estimators=400,
    random_state=42,
    n_jobs=-1,
)
rf.fit(X_scaled, df_scores["anomaly_score"].values)

explainer = shap.TreeExplainer(rf)

# Choose a window around the strongest detected event (or the global peak)
if not events_df.empty:
    # pick the max-score event
    i_best = int(events_df["anomaly_score_max"].values.argmax())
    t0 = pd.to_datetime(events_df.loc[i_best, "t_start"], utc=True)
    t1 = pd.to_datetime(events_df.loc[i_best, "t_end"], utc=True)
    pad = pd.Timedelta(minutes=10)
    w_start, w_end = t0 - pad, t1 + pad
    mask = (df_scores["time"] >= w_start) & (df_scores["time"] <= w_end)
    idxs = np.where(mask.values)[0]
else:
    peak_idx = int(np.argmax(df_scores["anomaly_score"].values))
    idxs = np.arange(max(0, peak_idx - 15), min(len(df_scores), peak_idx + 16))

# Cap time steps for readability
max_steps = 40
if len(idxs) > max_steps:
    idxs = np.linspace(idxs.min(), idxs.max(), max_steps).round().astype(int)

X_win = X_scaled[idxs]  # (time_steps, n_features)

# SHAP values for each time step
shap_vals = np.asarray(explainer.shap_values(X_win))  # (time_steps, n_features)

# Convert to (features, time_steps) for dashboard loader
shap_matrix = shap_vals.T
time_labels = [f"t{i}" for i in range(shap_matrix.shape[1])]

OUT_SHAP_CSV = FIG_DIR / "jamming_event_shap_values.csv"
save_shap_matrix_csv(OUT_SHAP_CSV, shap_matrix, feature_cols, time_labels=time_labels)


In [ ]:
# ============================================================
# 7) Save event heatmap PNG
# ============================================================
OUT_EVENT_PNG = FIG_DIR / "jamming_event_heatmap.png"

plt.figure(figsize=(12, 6))
plt.imshow(shap_matrix, aspect="auto")
plt.yticks(np.arange(len(feature_cols)), feature_cols, fontsize=7)
plt.xticks(np.arange(len(time_labels)), time_labels, rotation=90, fontsize=7)
plt.title("Jamming / Interference – SHAP heatmap around a high-score window (surrogate explanation)")
plt.tight_layout()
plt.savefig(OUT_EVENT_PNG, dpi=200, bbox_inches="tight")
plt.close()

print("Saved:", OUT_EVENT_PNG)


In [ ]:
# ============================================================
# 8) Continuous heatmap PNG (overview)
# ============================================================
OUT_CONT_PNG = FIG_DIR / "jamming_continuous_heatmap.png"

rng = np.random.default_rng(42)
n_windows = min(160, len(X_scaled))
sel = rng.choice(len(X_scaled), size=n_windows, replace=False)
sel = np.sort(sel)

X_sub = X_scaled[sel]
shap_sub = np.asarray(explainer.shap_values(X_sub))  # (n_windows, n_features)

# Select top-N features by mean absolute SHAP
mean_abs = np.mean(np.abs(shap_sub), axis=0)
top_n = min(18, len(feature_cols))
top_idx = np.argsort(mean_abs)[::-1][:top_n]

cont_matrix = shap_sub[:, top_idx].T  # (top_features, n_windows)
cont_feat_names = [feature_cols[i] for i in top_idx]
cont_time_labels = [f"w{i}" for i in range(n_windows)]

plt.figure(figsize=(12, 6))
plt.imshow(cont_matrix, aspect="auto")
plt.yticks(np.arange(len(cont_feat_names)), cont_feat_names, fontsize=7)
plt.xticks(np.arange(len(cont_time_labels))[::10], cont_time_labels[::10], rotation=90, fontsize=7)
plt.title("Jamming / Interference – Continuous SHAP overview (top features; surrogate explanation)")
plt.tight_layout()
plt.savefig(OUT_CONT_PNG, dpi=200, bbox_inches="tight")
plt.close()

print("Saved:", OUT_CONT_PNG)


## Completion check
Verify these outputs exist:
- `data/processed/jam_test_eventized_scores.csv`
- `reports/figures/jamming_event_shap_values.csv`
- `reports/figures/jamming_event_heatmap.png`
- `reports/figures/jamming_continuous_heatmap.png`

If the dashboard still falls back to PNGs, confirm the CSV path matches exactly.